# 🧠 Semana 6 - Dia 1: Deep Learning Intro

Notebook prático para acompanhar [docs/23-dia1-semana6-deep-learning-intro.md](../../docs/23-dia1-semana6-deep-learning-intro.md).

**Objetivos:**
1. Construir uma rede neural mínima do zero em NumPy (intuição de forward + loss).
2. Treinar uma NN com Keras em dataset tabular real.
3. Comparar NN vs RandomForest no mesmo problema.
4. Diagnosticar overfitting com curvas de loss e EarlyStopping.

**Pré-requisitos:** TensorFlow ≥ 2.10 instalado. Se não estiver, rode na primeira célula:
```
%pip install tensorflow
```

## 1. Setup

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
sns.set_theme(style='whitegrid')
print('Imports OK')

Imports OK


## 2. Mini-rede do zero (intuição)

Vamos construir um perceptron de 1 camada oculta usando só NumPy. Isso desmistifica o que Keras faz por baixo dos panos.

**Arquitetura:** entrada → Dense(8, ReLU) → Dense(1, Sigmoid)

In [2]:
def relu(x):
    return np.maximum(0, x)

def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

def bce_loss(y_true, y_pred, eps=1e-7):
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(y_true * np.log(y_pred) + (1 - y_true) * np.log(1 - y_pred))

# Dados de brinquedo (XOR-like nao-linear)
rng = np.random.default_rng(RANDOM_STATE)
X_toy = rng.normal(size=(200, 2))
y_toy = ((X_toy[:, 0] * X_toy[:, 1]) > 0).astype(int).reshape(-1, 1)

# Pesos iniciais aleatorios
n_in, n_hidden, n_out = 2, 8, 1
W1 = rng.normal(scale=0.5, size=(n_in, n_hidden))
b1 = np.zeros((1, n_hidden))
W2 = rng.normal(scale=0.5, size=(n_hidden, n_out))
b2 = np.zeros((1, n_out))

# Forward pass
z1 = X_toy @ W1 + b1
a1 = relu(z1)
z2 = a1 @ W2 + b2
y_pred = sigmoid(z2)

loss = bce_loss(y_toy, y_pred)
print(f'Forward pass shape: {y_pred.shape}')
print(f'Loss inicial (sem treinar): {loss:.4f}')
print('Esperado ~0.69 (= -log(0.5)) porque a rede saiu praticamente aleatoria.')

Forward pass shape: (200, 1)
Loss inicial (sem treinar): 0.8005
Esperado ~0.69 (= -log(0.5)) porque a rede saiu praticamente aleatoria.


> Esse foi só o **forward pass**. Treinar de verdade exige backpropagation, que é trabalhosa em NumPy puro. A partir daqui, deixamos o Keras cuidar disso.

## 3. Dataset real: Breast Cancer (sklearn)

30 features numéricas, target binário (maligno = 0, benigno = 1). Pequeno mas suficiente para exercitar o pipeline completo.

In [3]:
data = load_breast_cancer(as_frame=True)
df = data.frame.copy()
X = df.drop(columns=['target'])
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print(f'Train: {X_train_s.shape} | Test: {X_test_s.shape}')
print(f'Distribuicao do target (train):\n{y_train.value_counts(normalize=True).round(3)}')

Train: (455, 30) | Test: (114, 30)
Distribuicao do target (train):
target
1    0.626
0    0.374
Name: proportion, dtype: float64


## 4. Baseline: RandomForest

Antes de partir para a rede neural, fixe um baseline forte de ML clássico para ter referência.

In [4]:
rf = RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)  # RF nao precisa de scaler

rf_pred = rf.predict(X_test)
rf_proba = rf.predict_proba(X_test)[:, 1]

rf_acc = accuracy_score(y_test, rf_pred)
rf_auc = roc_auc_score(y_test, rf_proba)
print(f'RandomForest | Acc: {rf_acc:.4f} | AUC: {rf_auc:.4f}')

RandomForest | Acc: 0.9474 | AUC: 0.9937


## 5. Primeira NN com Keras

In [5]:
import os
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '2')  # silencia logs do TF

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.random.set_seed(RANDOM_STATE)
print(f'TensorFlow: {tf.__version__}')
print(f'GPU disponivel: {bool(tf.config.list_physical_devices("GPU"))}')

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
def build_model(n_features, hidden=(64, 32), dropout=0.2, lr=1e-3):
    model = keras.Sequential(name='nn_intro')
    model.add(layers.Input(shape=(n_features,)))
    for units in hidden:
        model.add(layers.Dense(units, activation='relu'))
        if dropout > 0:
            model.add(layers.Dropout(dropout))
    model.add(layers.Dense(1, activation='sigmoid'))
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy', keras.metrics.AUC(name='auc')]
    )
    return model

model = build_model(X_train_s.shape[1])
model.summary()

### Conta rápida de parâmetros

Camada 1: 30 × 64 + 64 = 1.984  
Camada 2: 64 × 32 + 32 = 2.080  
Saída : 32 × 1 + 1 = 33  
**Total ≈ 4.097 pesos para aprender.**

In [ ]:
early = keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=8, restore_best_weights=True, verbose=0
)
reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=4, min_lr=1e-5, verbose=0
)

history = model.fit(
    X_train_s, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early, reduce_lr],
    verbose=0
)
print(f'Treino terminou na epoca {len(history.history["loss"])}')

## 6. Curvas de aprendizado

Se train_loss << val_loss, há overfit. Se ambas estão altas, há underfit.

In [ ]:
h = history.history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(h['loss'], label='train')
axes[0].plot(h['val_loss'], label='val')
axes[0].set_title('Loss')
axes[0].set_xlabel('epoch'); axes[0].legend()

axes[1].plot(h['auc'], label='train')
axes[1].plot(h['val_auc'], label='val')
axes[1].set_title('AUC')
axes[1].set_xlabel('epoch'); axes[1].legend()
plt.tight_layout(); plt.show()

## 7. Avaliação no test set

In [ ]:
nn_proba = model.predict(X_test_s, verbose=0).ravel()
nn_pred = (nn_proba >= 0.5).astype(int)

nn_acc = accuracy_score(y_test, nn_pred)
nn_auc = roc_auc_score(y_test, nn_proba)
print(f'NN          | Acc: {nn_acc:.4f} | AUC: {nn_auc:.4f}')
print(f'RandomForest| Acc: {rf_acc:.4f} | AUC: {rf_auc:.4f}')

print('\nNN classification report:')
print(classification_report(y_test, nn_pred, digits=4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, name, pred in [(axes[0], 'NN', nn_pred), (axes[1], 'RF', rf_pred)]:
    cm = confusion_matrix(y_test, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, cbar=False)
    ax.set_title(f'Confusion Matrix - {name}')
    ax.set_xlabel('predito'); ax.set_ylabel('real')
plt.tight_layout(); plt.show()

## 8. Diagnóstico

Para esse dataset (569 amostras, 30 features, target bem separável), os dois modelos costumam ficar muito próximos. **Esse é um achado importante:** em tabular pequeno, NN não dá mágica.

**Quando a NN tende a vencer?**
- Mais dados (>50k linhas)
- Features brutas (imagem, áudio, texto)
- Estrutura espacial/temporal

**O que vale levar daqui:**
- Você sabe construir, treinar e avaliar uma NN com Keras.
- Você sabe ler curvas de loss para detectar overfit.
- Você sabe quando NÃO usar NN.

## 9. Exercício livre

1. Aumente uma camada (ex.: `(128, 64, 32)`) e refaça o treino.
2. Remova o Dropout e veja se aparece overfit.
3. Troque o otimizador para `SGD` com `learning_rate=1e-2` e compare.
4. Plote o histograma de probabilidades preditas (`nn_proba`) e veja se a rede está confiante.

Documente o que mudou no AUC e na curva de loss.